# Economy of Bits — Notebook 05: is it capital, or is it accounting?

Notebook 04 produced the result the paper will be built on, and four defects that must be
cleared before it can be reported.

**The result.** Firms in the top 1% of index value became sharply more capital-intensive
while every other group became lighter — asset lightness 0.845 → 0.647 (slope −0.019,
*p* < 0.0001) against +0.002 for other digital firms and +0.004 for the rest — and capex over
revenue in the tail went 0.088 → 0.168 with gross margin flat. Capital, not margins.

**The defects.** ExxonMobil is mapped to a 2025 re-registration with one year of filings
(BlackRock too), and it enters the tail group in 2025 alone at 0.333, inflating the very
slope being reported. Alphabet is split across GOOG and GOOGL as two broken series.
Berkshire has no PP&E at all. And from 2019, Amazon, Meta and Alphabet report PP&E under a
tag that folds in finance-lease right-of-use assets — **ASC 842** — switching in and out of
it mid-series.

The fix for the last one is a different measure, not a better one: **capital expenditure
over revenue** is a cash-flow quantity that lease capitalisation cannot touch. It becomes
the headline; asset lightness stays as corroboration with an explicit sensitivity test.

Upload `bits_inputs_v5.zip`. Runtime about 15 minutes.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 04 produced the result the paper will be built on, and four defects
# that have to be cleared before it can be reported.
#
# THE RESULT. Firms in the top 1% of index value became sharply more
# capital-intensive over the decade while every other group became lighter:
#
#     group     asset lightness 2016 -> 2025   slope/yr      p
#     tail          0.845 -> 0.647             -0.0190    <0.0001
#     digital       0.869 -> 0.893             +0.0022    <0.0001
#     rest          0.718 -> 0.747             +0.0040     0.0001
#
# and capital expenditure over revenue in the tail went 0.088 -> 0.168 while
# gross margin stayed flat. It is capital, not margins.
#
# THE DEFECTS.
#   1. ExxonMobil is mapped to CIK 2115436, a 2025 re-registration with one year
#      of filings. Its real history is CIK 34088. BlackRock has the same problem.
#      XOM enters the tail group in 2025 alone at 0.333, which drags the 2025
#      group mean down and inflates the very slope being reported.
#   2. Alphabet is split across GOOG and GOOGL, so it appears as two firms with
#      broken series. Firm identity has to key on CIK, not ticker.
#   3. ASC 842. From 2019, Amazon, Meta and Alphabet report PP&E under a tag that
#      INCLUDES finance-lease right-of-use assets, and they switch in and out of
#      it mid-series. Part of the fall in asset lightness for exactly the firms
#      the story is about could be lease capitalisation rather than compute.
#   4. Berkshire has no PP&E figure at all in ten years.
#
# The fix for (3) is not a better balance-sheet measure but a different one:
# CAPITAL EXPENDITURE OVER REVENUE is a cash-flow quantity, untouched by how
# leases are capitalised, and it measures the build-out directly. It becomes the
# headline in this notebook; asset lightness stays as corroboration with an
# explicit sensitivity test.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests
import statsmodels.api as sm
from scipy import stats as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 160)

RUN_TAG = "bits_05_capital"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE, FIGS = ROOT / "outputs", ROOT / "cache", ROOT / "outputs" / "figures"
for d in (OUTD, CACHE, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def cached(name, builder, refresh=False):
    p = CACHE / f"{name}.parquet"
    if p.exists() and not refresh:
        d = pd.read_parquet(p); print(f"  cache hit: {name} ({len(d):,} rows)"); return d
    d = builder()
    if d is not None and len(d):
        d.to_parquet(p); print(f"  cached {name} ({len(d):,} rows)")
    return d

SEC_HEADERS = {"User-Agent": "Boston University Metropolitan College research "
                             "(spattna1@bu.edu)", "Accept-Encoding": "gzip, deflate"}

def sec_get(url, tries=3, timeout=60):
    for a in range(tries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429):
                time.sleep(5 * (a + 1)); continue
            return None
        except Exception:
            if a == tries - 1:
                return None
            time.sleep(2 * (a + 1))
    return None

PAPER_YEARS = list(range(2016, 2026))
print(f"writing to: {ROOT}")

## 2 · Inputs, and the CIK corrections

In [ ]:
# =============================================================================
# CELL 2 - Inputs, and the two CIK corrections
#
# Detection rule, run before the overrides so it can find cases I have not
# thought of: a firm in the panel for at least eight years whose mapped CIK has
# fewer than five years of filings is mismapped. company_tickers.json gives the
# CURRENT registrant, and a firm that re-registers gets a CIK whose history
# starts at the reorganisation.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v5.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs5", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "panel_final.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v5.zip not found.")

rep = pd.read_csv(SRC / "panel_final.csv.gz"); rep["year"] = rep.year.astype(int)
facts = pd.read_csv(SRC / "edgar_facts_v2.csv.gz")
tick2cik = pd.read_csv(SRC / "ticker_to_cik.csv")
coding = pd.read_csv(SRC / "platform_coding_final.csv")
pub_ineq = pd.read_csv(SRC / "inequality_by_year.csv")
print(f"panel {len(rep):,} firm-years; facts {len(facts):,}")

CIK = dict(zip(tick2cik.ticker, tick2cik.cik))
rep["cik"] = rep.ticker.map(CIK)
panel_years = rep.groupby("ticker").year.nunique()
fact_years = facts[facts.field == "assets"].groupby("cik").year.nunique()
m = rep[["ticker", "cik"]].dropna().drop_duplicates()
m["panel_years"] = m.ticker.map(panel_years)
m["fact_years"] = m.cik.map(fact_years).fillna(0)
thin = m[(m.panel_years >= 8) & (m.fact_years < 5)].sort_values("fact_years")
print(f"\n{len(thin)} firms look mismapped:")
display(thin)

# Verified predecessor CIKs. Each was checked on EDGAR: the override is the
# entity that filed the 10-Ks covering the panel years.
CIK_OVERRIDE = {
    "XOM": 34088,     # Exxon Mobil Corp; 2115436 is the 2025 re-registration
    "BLK": 1364742,   # BlackRock, Inc.; 2012383 is the 2024 reorganisation
}
for t, c in CIK_OVERRIDE.items():
    if t in set(rep.ticker):
        rep.loc[rep.ticker == t, "cik"] = c
        print(f"  {t}: CIK {CIK.get(t)} -> {c}")
still = [t for t in thin.ticker if t not in CIK_OVERRIDE]
if still:
    print(f"\nSTILL MISMAPPED and not overridden: {still}")
    print("Look each of these up on EDGAR before trusting any firm-level series.")

# --- firm identity is the CIK, not the ticker --------------------------------
# Alphabet appears as GOOG in some years and GOOGL in others because the JRFM
# panel keeps whichever class was larger that year. As two tickers it is two
# broken series; as one CIK it is one firm with ten observations.
rep["firm_id"] = rep.cik.fillna(-1).astype(int).astype(str)
rep.loc[rep.cik.isna(), "firm_id"] = rep.loc[rep.cik.isna(), "ticker"]
name = (rep.sort_values("mktcap", ascending=False).groupby("firm_id").ticker.first())
rep["firm"] = rep.firm_id.map(name)
dupe = rep.groupby("firm_id").ticker.nunique()
print(f"\n{int((dupe > 1).sum())} firms carry more than one ticker; "
      f"{rep.ticker.nunique()} tickers collapse to {rep.firm_id.nunique()} firms")
for fid in dupe[dupe > 1].index[:6]:
    sub = rep[rep.firm_id == fid]
    print(f"  {name[fid]}: {sorted(sub.ticker.unique())}")
rep["cap_share"] = rep.mktcap / rep.groupby("year").mktcap.transform("sum")

## 3 · Refetch, recording which PP&E tag each figure came from

In [ ]:
# =============================================================================
# CELL 3 - Refetch the corrected CIKs, and record which PP&E tag was used
#
# The tag matters this time. ASC 842 took effect for large filers in 2019 and
# moved operating leases onto the balance sheet. Amazon, Meta and Alphabet
# report PP&E under a tag that folds in finance-lease right-of-use assets, and
# they move in and out of it mid-series. Every firm-year now carries the tag it
# came from so cell 6 can test how much of the fall in asset lightness is
# leases rather than compute.
# =============================================================================
CONCEPTS = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax",
                "RevenueFromContractWithCustomerIncludingAssessedTax",
                "Revenues", "SalesRevenueNet", "SalesRevenueGoodsNet"],
    "cogs": ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold",
             "CostOfServices"],
    "gross_profit": ["GrossProfit"],
    "rnd": ["ResearchAndDevelopmentExpense"],
    "operating_income": ["OperatingIncomeLoss"],
    "ppe_net": ["PropertyPlantAndEquipmentNet",
                "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization",
                "RealEstateInvestmentPropertyNet"],
    "ppe_gross": ["PropertyPlantAndEquipmentGross"],
    "assets": ["Assets"],
    "goodwill": ["Goodwill"],
    "intangibles": ["IntangibleAssetsNetExcludingGoodwill", "FiniteLivedIntangibleAssetsNet"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment",
              "PaymentsToAcquireProductiveAssets"],
    "depreciation": ["DepreciationDepletionAndAmortization",
                     "DepreciationAmortizationAndAccretionNet"],
    "operating_lease_rou": ["OperatingLeaseRightOfUseAsset"],
}
STOCKS = {"ppe_net", "ppe_gross", "assets", "goodwill", "intangibles",
          "operating_lease_rou"}

def parse_facts(js, cik):
    out = []
    gaap = (js or {}).get("facts", {}).get("us-gaap", {})
    for field, tags in CONCEPTS.items():
        is_stock = field in STOCKS
        for rank, tag in enumerate(tags):
            node = gaap.get(tag)
            if not node:
                continue
            for unit, obs in node.get("units", {}).items():
                if unit != "USD":
                    continue
                for o in obs:
                    if o.get("form") not in ("10-K", "10-K/A", "20-F", "40-F"):
                        continue
                    if o.get("val") is None or not o.get("end"):
                        continue
                    if is_stock:
                        year = pd.Timestamp(o["end"]).year
                    else:
                        if not o.get("start"):
                            continue
                        days = (pd.Timestamp(o["end"]) - pd.Timestamp(o["start"])).days
                        if not (330 <= days <= 400):
                            continue
                        e = pd.Timestamp(o["end"])
                        year = int((e - pd.Timedelta(days=183)).year)
                    out.append({"cik": int(cik), "year": year, "field": field,
                                "tag": tag, "tag_rank": rank, "value": float(o["val"]),
                                "end": o["end"], "accn": o.get("accn"),
                                "source": "companyfacts"})
    return out

known = set(facts.cik.dropna().astype(int))
NEED = sorted(set(rep.cik.dropna().astype(int)) - known
              | set(CIK_OVERRIDE.values()))
# Anyone missing capex or gross PP&E, which notebook 04 collected only partly.
have = facts.pivot_table(index=["cik", "year"], columns="field",
                         values="value", aggfunc="first").reset_index()
for c in ("capex", "ppe_net"):
    if c not in have.columns:
        have[c] = np.nan
NEED = sorted(set(NEED) | set(have.loc[have.capex.isna() | have.ppe_net.isna(), "cik"]
                              .dropna().astype(int))
              & set(rep.cik.dropna().astype(int)))
print(f"{len(NEED)} CIKs to fetch  (~{len(NEED)*0.5/60:.0f} min)")

def fetch(ciks):
    rows, miss = [], 0
    for i, c in enumerate(ciks, 1):
        js = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(c):010d}.json")
        if js is None:
            miss += 1
        else:
            rows.extend(parse_facts(js, c))
        if i % 50 == 0:
            print(f"    {i}/{len(ciks)}, {len(rows):,} facts, {miss} unavailable")
    return pd.DataFrame(rows)

new_facts = cached("companyfacts_v3", lambda: fetch(NEED))
old = facts.copy()
old["tag_rank"] = [CONCEPTS.get(f, [t]).index(t) if t in CONCEPTS.get(f, []) else 99
                   for f, t in zip(old.field, old.tag)]
old["end"] = pd.NaT; old["accn"] = ""
allf = pd.concat([old, new_facts], ignore_index=True) if len(new_facts) else old
allf["src_rank"] = (allf.source == "frames").astype(int)
allf["end"] = pd.to_datetime(allf.get("end"), errors="coerce")
resolved = (allf.sort_values(["cik", "year", "field", "tag_rank", "src_rank", "end", "accn"],
                             ascending=[True, True, True, True, True, False, False])
            .groupby(["cik", "year", "field"], as_index=False)
            .first()[["cik", "year", "field", "value", "tag", "source"]])
print(f"\n{len(resolved):,} resolved firm-year-fields, {resolved.cik.nunique()} CIKs")
savetable(resolved, "edgar_facts_v3")

## 4 · Components at the firm level

In [ ]:
# =============================================================================
# CELL 4 - Components, at the firm level, with the tag carried through
# =============================================================================
val = resolved.pivot_table(index=["cik", "year"], columns="field",
                           values="value", aggfunc="first").reset_index()
val.columns.name = None
tag = (resolved[resolved.field == "ppe_net"][["cik", "year", "tag"]]
       .rename(columns={"tag": "ppe_tag"}))
val = val.merge(tag, on=["cik", "year"], how="left")
for c in list(CONCEPTS) + ["ppe_tag"]:
    if c not in val.columns:
        val[c] = np.nan

F = rep.merge(val, on=["cik", "year"], how="left")
F["gross_profit"] = F.gross_profit.fillna(F.revenue - F.cogs)
F["asset_light"] = 1 - (F.ppe_net / F.assets)
F["capex_intensity"] = F.capex / F.revenue          # lease-immune, the headline
F["ppe_share"] = F.ppe_net / F.assets
F["dep_intensity"] = F.depreciation / F.revenue
F["gross_margin"] = F.gross_profit / F.revenue
F["rnd_intensity"] = F.rnd / F.revenue
F["intangible_share"] = F.intangibles / F.assets
F["lease_tag"] = F.ppe_tag.astype(str).str.contains("FinanceLease")
MEASURES = ["capex_intensity", "asset_light", "ppe_share", "dep_intensity",
            "gross_margin", "rnd_intensity", "intangible_share"]
for c in MEASURES:
    lo = F.groupby("year")[c].transform(lambda s: s.quantile(0.01))
    hi = F.groupby("year")[c].transform(lambda s: s.quantile(0.99))
    F[c] = F[c].clip(lo, hi)

print("coverage:")
for c in MEASURES:
    print(f"  {c:<18}{100*F[c].notna().mean():5.1f}%")
print(f"\nfirm-years on the lease-inclusive PP&E tag: "
      f"{int(F.lease_tag.sum())} ({100*F.lease_tag.mean():.1f}%)")
print("by year:")
print(F.groupby("year").lease_tag.sum().astype(int).to_string())

# --- groups, fixed on the corrected firm identity ----------------------------
DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
F["gics_digital"] = (F.sector.isin(DIGITAL_SECTORS)
                     | F.ticker.str.upper().isin(DIGITAL_EXTRA))
top1 = F.groupby("year").cap_share.transform(lambda s: s >= s.quantile(0.99))
TAIL = set(F.loc[top1, "firm_id"].unique())
F["group3"] = np.where(F.firm_id.isin(TAIL), "tail",
                       np.where(F.gics_digital, "digital", "rest"))
# The four firms that operate hyperscale compute, as against the tail firms that
# design chips or sell devices. The distinction is the mechanism: compute sits
# on the balance sheet of whoever RUNS it. Nvidia is fabless and Apple
# outsources assembly, so neither should show the build-out if the story holds.
HYPERSCALERS = {"MSFT", "GOOGL", "GOOG", "AMZN", "META"}
F["hyperscaler"] = F.ticker.isin(HYPERSCALERS)
print(f"\ntail firms: {sorted(F.loc[F.group3=='tail','firm'].unique())}")
print(f"hyperscalers: {sorted(F.loc[F.hyperscaler,'firm'].unique())}")

# Data completeness per tail firm, since notebook 04's group means were moved by
# firms entering the panel late rather than by anything economic.
print("\nyears of data per tail firm:")
tt = F[F.group3 == "tail"]
display(tt.groupby("firm")[["capex_intensity", "asset_light"]]
        .apply(lambda d: d.notna().sum()).sort_values("capex_intensity"))
savetable(F[["year", "firm_id", "firm", "ticker", "cik", "sector", "mktcap",
             "cap_share", "group3", "hyperscaler", "gics_digital", "lease_tag",
             "ppe_tag"] + MEASURES], "firm_measures_v3")

## 5 · The headline, on a balanced panel

Everything twice: all firms, and firms with a complete ten-year series. The gap between
them is the size of the composition problem.

In [ ]:
# =============================================================================
# CELL 5 - The headline, on a BALANCED panel
#
# Notebook 04 reported group means over whatever firms happened to have data
# that year. Exxon entered the tail group in 2025 alone at an asset lightness of
# 0.333 and pulled the group mean down; part of the slope was composition, not
# behaviour.
#
# Everything here is computed twice: on the unbalanced panel, and on firms with
# a complete ten-year series. If the two disagree the balanced version is the
# one to believe, and the gap between them is the size of the composition
# problem.
# =============================================================================
def trends(df, var, by="group3", balanced=False):
    d = df.dropna(subset=[var]).copy()
    if balanced:
        n = d.groupby("firm_id").year.nunique()
        keep = set(n[n == len(PAPER_YEARS)].index)
        d = d[d.firm_id.isin(keep)]
    piv = d.pivot_table(index="year", columns=by, values=var)
    rows = []
    for g in piv.columns:
        s = piv[g].dropna()
        if len(s) < 5:
            continue
        X = sm.add_constant(s.index.values - np.mean(s.index.values))
        fit = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
        rows.append({"variable": var, "panel": "balanced" if balanced else "all",
                     "group": g, "n_firms": int(d.loc[d[by] == g, "firm_id"].nunique()),
                     "first": round(float(s.iloc[0]), 4),
                     "last": round(float(s.iloc[-1]), 4),
                     "slope": round(float(fit.params[1]), 5),
                     "p_hac": round(float(fit.pvalues[1]), 4)})
    return piv, pd.DataFrame(rows)

out = []
for var in ["capex_intensity", "asset_light", "gross_margin", "dep_intensity"]:
    print("=" * 72)
    print(var.upper())
    print("=" * 72)
    for bal in (False, True):
        piv, t = trends(F, var, balanced=bal)
        print(f"\n{'balanced (complete 10-year firms only)' if bal else 'all firms'}:")
        display(piv.round(4))
        display(t)
        out.append(t)
trend_all = pd.concat(out, ignore_index=True)
savetable(trend_all, "trends_by_group")

print("\n" + "=" * 72)
print("WHAT TO CHECK")
print("=" * 72)
print("capex_intensity is the measure to lead on. It is a cash-flow quantity,")
print("so ASC 842 cannot touch it, and it measures the build-out directly.")
print("If the tail's capex intensity rises on BOTH panels while the other two")
print("groups do not, the finding is real and is not composition.")
print()
print("Compare the balanced and unbalanced slopes for asset_light. Notebook 04")
print("reported -0.0190 on the unbalanced panel with Exxon entering in 2025 at")
print("0.333. If the balanced slope is much smaller, that number was partly an")
print("artefact and the paper must quote the balanced one.")

## 6 · Compute or lease accounting?

Three tests, because this is the first objection a referee will raise — then the sharpest
test of the mechanism: hyperscalers against the rest of the tail.

In [ ]:
# =============================================================================
# CELL 6 - Is it compute, or is it lease accounting?
#
# Three tests, because this is the objection a referee will raise first.
#
#   1. Capex is a cash-flow measure. Lease capitalisation does not appear in it
#      at all. If capex intensity rises with the balance-sheet measure, leases
#      cannot be the explanation.
#   2. Restrict to firm-years reported under the PLAIN PropertyPlantAndEquipment
#      Net tag and re-estimate. If the fall survives on that subsample it is not
#      the tag.
#   3. Test for a break at 2019, when ASC 842 took effect. Lease capitalisation
#      is a step; a compute build-out is a trend.
# =============================================================================
tail = F[F.group3 == "tail"].dropna(subset=["asset_light"])
print("test 1 -- do the cash-flow and balance-sheet measures move together?")
tp = F[F.group3 == "tail"].groupby("year")[["capex_intensity", "asset_light"]].mean()
display(tp.round(4))
r = tp.corr().iloc[0, 1]
print(f"corr(capex intensity, asset lightness) across years = {r:+.3f}")
print("  A strongly NEGATIVE correlation means the two independent measures")
print("  agree: more capital spending, less asset-light. Leases move only one.")

print("\ntest 2 -- restrict to the plain PP&E tag")
plain = tail[~tail.lease_tag]
print(f"  {len(plain)} of {len(tail)} tail firm-years use the plain tag")
piv = plain.pivot_table(index="year", values="asset_light", aggfunc="mean")
display(piv.round(4))
s = piv.asset_light.dropna()
if len(s) >= 5:
    X = sm.add_constant(s.index.values - np.mean(s.index.values))
    f2 = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    print(f"  slope {float(f2.params[1]):+.5f}/yr, p = {float(f2.pvalues[1]):.4f}")
    print("  compare with the all-tags slope reported in cell 5.")

print("\ntest 3 -- is there a step at 2019?")
d = tail.copy()
d["post"] = (d.year >= 2019).astype(float)
d["t"] = d.year - 2019
X = sm.add_constant(pd.DataFrame({"t": d.t, "post": d.post, "post_t": d.post * d.t}))
f3 = sm.OLS(d.asset_light.values, X.values.astype(float)).fit(
    cov_type="cluster", cov_kwds={"groups": d.firm_id.values})
print(f"  pre-2019 slope        {float(f3.params[1]):+.5f}  p = {float(f3.pvalues[1]):.4f}")
print(f"  step at 2019          {float(f3.params[2]):+.5f}  p = {float(f3.pvalues[2]):.4f}")
print(f"  change in slope after {float(f3.params[3]):+.5f}  p = {float(f3.pvalues[3]):.4f}")
print("  A significant STEP with no slope change is lease accounting. A slope")
print("  that is negative before 2019 and steeper after is a build-out that")
print("  accelerated, which is the compute story.")
savetable(pd.DataFrame([{"test": "corr capex vs asset_light", "value": round(float(r), 4)},
                        {"test": "plain-tag slope", "value": round(float(f2.params[1]), 5),
                         "p": round(float(f2.pvalues[1]), 4)},
                        {"test": "2019 step", "value": round(float(f3.params[2]), 5),
                         "p": round(float(f3.pvalues[2]), 4)},
                        {"test": "2019 slope change", "value": round(float(f3.params[3]), 5),
                         "p": round(float(f3.pvalues[3]), 4)}]), "asc842_sensitivity")

# --- the sharpest test of the mechanism --------------------------------------
print("\n" + "=" * 72)
print("HYPERSCALERS AGAINST THE REST OF THE TAIL")
print("=" * 72)
print("Compute sits on the balance sheet of whoever RUNS it. Nvidia is fabless")
print("and Apple outsources assembly, so if the mechanism is compute rather than")
print("something general about large firms, the build-out should appear in")
print("Microsoft, Alphabet, Amazon and Meta and NOT in Nvidia or Apple.")
sub = F[F.group3 == "tail"].copy()
sub["kind"] = np.where(sub.hyperscaler, "hyperscaler", "other tail firm")
for var in ["capex_intensity", "asset_light"]:
    piv = sub.pivot_table(index="year", columns="kind", values=var)
    print(f"\n{var}:")
    display(piv.round(4))
print("\nfirm by firm, capital expenditure over revenue:")
display(sub.pivot_table(index="year", columns="firm", values="capex_intensity").round(3))
print("\nfirm by firm, asset lightness:")
display(sub.pivot_table(index="year", columns="firm", values="asset_light").round(3))
savetable(sub[["year", "firm", "kind", "capex_intensity", "asset_light"]],
          "tail_firm_detail")

## 7 · Why H1 inverted — bit intensity, or pharmaceuticals?

In [ ]:
# =============================================================================
# CELL 7 - Why H1 inverted
#
# Notebook 04's Gabaix-Ibragimov regression found that high-bit-intensity firms
# have a THINNER tail, +0.21 at p = 0.0015. That is H1 rejected in the opposite
# direction, and before the paper reports it as a finding it has to be checked
# for the obvious alternative: composition.
#
# Bit intensity ranks Health Care highest. Pharmaceuticals are numerous and
# mid-sized, with no firm near the top of the index. If the high-bit-intensity
# group is mostly pharma, its thin tail says something about pharmaceutical
# company sizes and nothing about the economics of bits.
#
# The test is to rank firms only against their own sector. If the inversion
# survives, it is about bit intensity; if it disappears, it was pharma.
# =============================================================================
Z = F.copy()
BI_COMPONENTS = ["intangible_share", "rnd_intensity", "gross_margin"]
for c in BI_COMPONENTS:
    Z[f"z_{c}"] = Z.groupby("year")[c].transform(
        lambda s: (s - s.mean()) / s.std() if s.std() and s.notna().sum() > 5 else np.nan)
Z["BI"] = Z[[f"z_{c}" for c in BI_COMPONENTS]].mean(axis=1)
Z.loc[Z[BI_COMPONENTS].notna().sum(axis=1) < 2, "BI"] = np.nan
Z["BI"] = Z.groupby("year").BI.transform(lambda s: (s - s.mean()) / s.std())
Z["BI_sector_neutral"] = Z.groupby(["year", "sector"]).BI.transform(
    lambda s: (s - s.mean()) / s.std() if s.notna().sum() > 5 else np.nan)
print(f"BI: {int(Z.BI.notna().sum()):,} firm-years; "
      f"sector-neutral: {int(Z.BI_sector_neutral.notna().sum()):,}")
print(f"corr(BI, sector-neutral BI) = "
      f"{Z[['BI','BI_sector_neutral']].corr().iloc[0,1]:+.3f}")

print("\nwhat sits in the top bit-intensity decile?")
hi = Z[Z.BI >= Z.groupby("year").BI.transform(lambda s: s.quantile(0.90))]
comp = (hi.sector.value_counts(normalize=True).head(6).rename("share_of_top_decile")
        .to_frame().round(3))
comp["share_of_panel"] = Z.sector.value_counts(normalize=True).reindex(comp.index).round(3)
display(comp)
print("If Health Care is far over-represented, the inversion is a statement")
print("about pharmaceutical firm sizes.")

def gi(df, var, frac=0.20):
    d = df.dropna(subset=[var, "cap_share"]).copy()
    d["high"] = d.groupby("year")[var].transform(
        lambda s: (s >= s.median()).astype(float))
    parts = []
    for (y, h), g in d.groupby(["year", "high"]):
        g = g.sort_values("cap_share", ascending=False).head(
            max(15, int(frac * len(g)))).copy()
        g["rank"] = np.arange(1, len(g) + 1)
        parts.append(g)
    dd = pd.concat(parts)
    X = sm.add_constant(pd.DataFrame({
        "log_size": np.log(dd.cap_share), "high": dd.high,
        "inter": dd.high * np.log(dd.cap_share)}))
    fit = sm.OLS(np.log(dd["rank"] - 0.5).values, X.values.astype(float)).fit(
        cov_type="cluster", cov_kwds={"groups": dd.firm_id.values})
    b, dc = float(fit.params[1]), float(fit.params[3])
    return {"ranked_on": var, "n": len(dd), "alpha_low": round(-b, 4),
            "alpha_high": round(-(b + dc), 4), "difference": round(-dc, 4),
            "p": round(float(fit.pvalues[3]), 4)}

rows = [gi(Z, "BI"), gi(Z, "BI_sector_neutral")]
res = pd.DataFrame(rows)
print("\nGabaix-Ibragimov, ranked on BI and on sector-neutral BI:")
display(res)
print("\nH1 predicted a negative difference. Notebook 04 found +0.21 on the")
print("raw index. If the sector-neutral row is near zero, the inversion was")
print("sector composition and the honest statement is that bit intensity is")
print("UNRELATED to tail position, not that it is inversely related.")
savetable(res, "h1_inversion_diagnosis")
savetable(Z[["year", "firm_id", "firm", "sector", "cap_share", "BI",
             "BI_sector_neutral", "group3"]], "bit_intensity_v3")

## 8 · Figures and download

In [ ]:
# =============================================================================
# CELL 8 - Figures and the results tables the paper will quote
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY = "#1f3b57", "#c8702a", "#7a7a7a"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
for ax, var, ttl in [(axes[0], "capex_intensity", "Capital expenditure / revenue"),
                     (axes[1], "asset_light", "Asset lightness (1 - PP&E/assets)")]:
    piv = F.pivot_table(index="year", columns="group3", values=var)
    for col, c in zip(["tail", "digital", "rest"], [INK, ACC, GREY]):
        if col in piv.columns:
            ax.plot(piv.index, piv[col], "o-", color=c, lw=2, ms=4, label=col)
    ax.set_xlabel("year"); ax.set_title(ttl); ax.legend(fontsize=8)

ax = axes[2]
sub = F[F.group3 == "tail"]
pv = sub.pivot_table(index="year", columns="firm", values="capex_intensity")
cm = plt.cm.viridis(np.linspace(0.1, 0.9, pv.shape[1]))
for c, col in zip(cm, pv.columns):
    ax.plot(pv.index, pv[col], "o-", color=c, lw=1.6, ms=3, label=str(col))
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("Firm by firm inside the tail"); ax.legend(fontsize=6.5, ncol=2)
fig.suptitle("Notebook 05: the tail is buying capital",
             fontsize=10, color=GREY, y=1.02)
savefig(fig, "C1_capital_intensity")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows); manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "cik_overrides": CIK_OVERRIDE, "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  trends_by_group.csv       balanced vs unbalanced, all four measures")
print("  asc842_sensitivity.csv    whether leases explain the balance-sheet fall")
print("  tail_firm_detail.csv      hyperscalers against the rest of the tail")
print("  h1_inversion_diagnosis.csv  whether the H1 reversal was pharma")
print()
print("If capex intensity rises in the tail on the balanced panel, survives the")
print("lease tests, and is concentrated in the four hyperscalers rather than")
print("spread across all large firms, the empirical work is finished and the")
print("next step is writing.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")